# Partner Ranking with Groq Extraction + Gaussian Scoring

This notebook ranks partners by combining:
- Text/service relevance from embeddings
- Geographic proximity from haversine distance
- Emergency-service boosting

It uses an async Groq client to extract symptoms and possible related services from user text.


In [1]:
import asyncio
import json
import math
import re
from pathlib import Path

import numpy as np
from groq import AsyncGroq
from sentence_transformers import SentenceTransformer


/Users/javi.fong/Documents/wais/agexport/agexport-smart-directory/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GROQ_MODEL = "openai/gpt-oss-120b"

_service_candidates = [Path("services.json"), Path("local test/services.json")]
_partner_candidates = [Path("partners.json"), Path("local test/partners.json")]

SERVICES_PATH = next((p for p in _service_candidates if p.exists()), _service_candidates[0])
PARTNERS_PATH = next((p for p in _partner_candidates if p.exists()), _partner_candidates[0])

_model = SentenceTransformer(MODEL_NAME)

with SERVICES_PATH.open("r", encoding="utf-8") as f:
    services_data = json.load(f)

with PARTNERS_PATH.open("r", encoding="utf-8") as f:
    partners_data = json.load(f)

service_embedding_map = {}
for item in services_data:
    service_name = str(item.get("og_service_name", "")).strip().lower()
    emb = item.get("embedding")
    if service_name and isinstance(emb, list) and emb:
        service_embedding_map[service_name] = np.asarray(emb, dtype=np.float32)

EMERGENCY_KEYWORDS = (
"emergencias dentales en niños"
, "emergencias dentales a niños"
, "emergencias dentales"
, "emergencias"
)


/Users/javi.fong/Documents/wais/agexport/agexport-smart-directory/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
def _parse_lat_lon(lat_lon):
    if isinstance(lat_lon, str):
        parts = [p.strip() for p in lat_lon.split(",")]
        if len(parts) != 2:
            raise ValueError("lat_lon string must be in 'lat,lon' format")
        return float(parts[0]), float(parts[1])

    if isinstance(lat_lon, (list, tuple)) and len(lat_lon) == 2:
        return float(lat_lon[0]), float(lat_lon[1])

    raise ValueError("lat_lon must be a 'lat,lon' string, list, or tuple with 2 values")


def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def _gaussian_similarity(similarity: float, sigma: float = 0.3) -> float:
    return float(math.exp(-0.5 * ((similarity - 1.0) / sigma) ** 2))


def _gaussian_distance_km(distance_km: float, sigma_km: float = 50.0) -> float:
    return float(math.exp(-0.5 * (distance_km / sigma_km) ** 2))


def _haversine_km(lat1, lon1, lat2, lon2) -> float:
    r = 6371.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = math.sin(dlat / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return r * c


def _safe_json_parse(content: str):
    content = (content or "").strip()
    if not content:
        return None

    try:
        return json.loads(content)
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*\}", content)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None

    return None


def _has_emergency_signal(text_items):
    return any(k in text_items for k in EMERGENCY_KEYWORDS)


In [4]:
async def extract_symptoms_services_with_groq(
    text: str,
    groq_client: AsyncGroq | None = None,
    model: str = GROQ_MODEL,
):
    if not text or not str(text).strip():
        return {"symptoms": [], "possible_services": [], "is_emergency": False}

    client = groq_client or AsyncGroq()

    completion = await client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract medical intent from user text: \
{\"symptoms\": string[], \"possible_services\": string[]}. \
Rules: \
- symptoms: medical symptoms/conditions/injuries mentioned or implied. \
- possible_services: likely healthcare services related to the case. all fractures are just emergencies. \
- is_emergency: true if urgency/emergency is indicated. all fractures are just emergencies. if bleeding is involved is an emergency. \
- Keep outputs concise and in Spanish if user text is Spanish."
                ),
            },
            {"role": "user", "content": str(text)},
        ],
        temperature=0,
        top_p=1,
        stream=False,
        response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "specialty_mapping",
                        "strict": True,
                        "schema": {
                            "type": "object",
                            "properties": {
                                "symptoms": {
                                    "type": "array",
                                    "items": {"type": "string"}
                                },
                                "possible_services": {
                                    "type": "array",
                                    "items": {"type": "string"}
                                },
                                "is_emergency": {
                                     "type": "boolean",
                                }
                            },
                            "required": ["symptoms", "possible_services", "is_emergency"],
                            "additionalProperties": False
                        }
                    }
                }
    )

    raw = completion.choices[0].message.content if completion.choices else ""
    data = _safe_json_parse(raw) or {}

    symptoms = data.get("symptoms") if isinstance(data, dict) else []
    services = data.get("possible_services") if isinstance(data, dict) else []
    is_emergency = data.get("is_emergency") if isinstance(data, dict) else False

    symptoms = [str(x).strip() for x in (symptoms or []) if str(x).strip()]
    services = [str(x).strip() for x in (services or []) if str(x).strip()]

    emergency_fallback = _has_emergency_signal([text] + symptoms + services)

    return {
        "symptoms": symptoms,
        "possible_services": services,
        "is_emergency" : is_emergency
    }


In [ ]:
async def rank_partners(
    text: str,
    lat_lon,
    top_k: int = 10,
    service_weight: float = 0.8,
    distance_weight: float = 0.2,
    similarity_sigma: float = 0.3,
    distance_sigma_km: float = 20.0,
    emergency_service_boost: float = 0.20,
    emergency_case_extra_boost: float = 0.20,
    groq_client: AsyncGroq | None = None,
    groq_model: str = GROQ_MODEL,
):
    if not text or not str(text).strip():
        raise ValueError("text cannot be empty")

    if service_weight < 0 or distance_weight < 0:
        raise ValueError("weights must be >= 0")

    if service_weight + distance_weight == 0:
        raise ValueError("at least one weight must be > 0")

    lat, lon = _parse_lat_lon(lat_lon)

    extracted = await extract_symptoms_services_with_groq(
        text=text,
        groq_client=groq_client,
        model=groq_model,
    )

    query_chunks = [str(text)] + extracted["symptoms"] + extracted["possible_services"]
    # query_chunks = [str(text)] + extracted["possible_services"]
    query_chunks = [q for q in query_chunks if q and str(q).strip()]
    query_vectors = _model.encode(query_chunks, convert_to_numpy=True).astype(np.float32)
    if query_vectors.ndim == 1:
        query_emb = query_vectors
    else:
        query_emb = np.mean(query_vectors, axis=0).astype(np.float32)

    ranked = []

    for partner in partners_data:
        p_name = partner.get("partner_name")
        partner_services = [str(s).strip().lower() for s in partner.get("partner_services", [])]

        service_scores = []
        for service_name in partner_services:
            emb = service_embedding_map.get(service_name)
            if emb is None:
                continue

            raw_similarity = _cosine_similarity(query_emb, emb)
            sim_score = _gaussian_similarity(raw_similarity, sigma=similarity_sigma)

            if _has_emergency_signal([service_name]) and extracted['is_emergency']:
                sim_score_new = sim_score * 3 #(1 + emergency_service_boost)
                print(f"{service_name} at {partner['partner_name']} boosted from {sim_score} to {sim_score_new}")
                sim_score = sim_score_new

            service_scores.append(sim_score)

        if service_scores:
            print(f"{p_name} - {partner_services[service_scores.index(max(service_scores))]}")
            top_scores = sorted(service_scores, reverse=True)[:1]
            service_score = float(sum(top_scores) / len(top_scores))
        else:
            service_score = 0.0

        best_location = None
        best_location_score = -1.0

        for idx, geo in enumerate(partner.get("partner_geo_locations", [])):
            if not isinstance(geo, dict):
                continue

            glat = geo.get("lat")
            glon = geo.get("lon")
            if glat is None or glon is None:
                continue

            try:
                distance_km = _haversine_km(lat, lon, float(glat), float(glon))
            except (TypeError, ValueError):
                continue

            distance_score = _gaussian_distance_km(distance_km, sigma_km=distance_sigma_km)
            
            combined_score = service_score
            # combined_score = 0.8 * service_score + 0.2 * distance_score
            # combined_score = service_score * distance_score
            

            location_result = {
                "location_index": idx,
                "query": geo.get("query"),
                "name": geo.get("name"),
                "address": geo.get("address"),
                "lat": float(glat),
                "lon": float(glon),
                "maps_url": geo.get("maps_url"),
                "distance_km": float(distance_km),
                "distance_score": float(distance_score),
                "combined_score": float(combined_score),
            }

            if combined_score > best_location_score:
                best_location_score = combined_score
                best_location = location_result

        if best_location is None:
            combined_score = service_weight * service_score
            best_location = {
                "location_index": None,
                "query": None,
                "name": None,
                "address": None,
                "lat": None,
                "lon": None,
                "maps_url": None,
                "distance_km": None,
                "distance_score": 0.0,
                "combined_score": float(combined_score),
            }

        ranked.append({
            "partner_name": partner.get("partner_name"),
            "partner_category": partner.get("partner_category"),
            "service_score": float(service_score),
            "final_score": float(best_location["combined_score"]),
            "best_location": best_location,
            "location_distance_km": best_location["distance_km"],
            "partner_phone_number": partner.get("partner_phone_number", []),
            "partner_whatsapp": partner.get("partner_whatsapp", []),
            "extracted_signals": extracted,
        })

    ranked.sort(key=lambda x: x["final_score"], reverse=True)
    return ranked[:top_k]


def rank_partners_sync(*args, **kwargs):
    return asyncio.run(rank_partners(*args, **kwargs))


In [42]:
# Example (Jupyter supports top-level await)
message = "tengo nauseas y me duele el brazo izquierdo"
results = await rank_partners(message, "15.78347, -90.23076", top_k=2)
for _r in results: 
    print(f"{_r['partner_name']} at {_r['best_location']['distance_km']}. {_r['best_location']['maps_url']}")

Centro Dental - dolor mandibular
Centro Dental San Lucas - cirugía maxilofacial
GrupoDent - cirugía maxilofacial
Facultad Odontología UFM - emergencias dentales
Clidenth - emergencias dentales
Centro Dental Kyrios - cirugía maxilofacial
Ríe Genial - prótesis totales
Vitalmed Odontología - emergencias dentales
Clínicas Sonríe - servicios dentales
Dental Core - odontología integral y especializada
G SMILE - emergencias dentales
Duo Dental Group - odontopediatría
ISMILE - odontolodía general
DENTIS - emergencias dentales
Clínicas La Guardía - diagnóstico radiológico
Clínica Dental Odontoafre - emergencias dentales
Nugenesis Plastic Surgery - cirugía de busto
Elán Med Center - sueroterapía
VISUALIZA - segmento anterior
Centro Integral de Rehabilitación REHAB - masoterapía
Integra Cáncer Institute - cirugía
Advance Urología - tratamiento quirurgico
Hospital Herrera Llerandi - emergencias
Hospital Centro Médico - emergencias
Hospital Ángeles - emergencias
MEDCARE - cardiología
ATENUN 
Atenci